# LAB 5- Apply PCA and t-SNE for dimensionality reduction. Visualize high-dimensional data in 2D. Measure accuracy before and after feature reduction.

# Dimensionality Reduction and k-NN Classification

This notebook explores how reducing high-dimensional data impacts machine learning performance and visualization. We will use the **Breast Cancer Wisconsin Dataset (30 features)**.

We will compare three scenarios:
1. **Baseline**: k-NN on the original 30-dimensional data.
2. **PCA (Principal Component Analysis)**: A linear technique that maximizes variance.
3. **t-SNE (t-Distributed Stochastic Neighbor Embedding)**: A non-linear technique excellent for preserving local data structures in lower dimensions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Load and scale the dataset
student = pd.read_csv("Student Depression Dataset(2).csv")
X = pd.get_dummies(
    student.drop(columns=["Depression", "id"]),
    drop_first=True,
    dtype=float
).fillna(0).to_numpy()
y = student["Depression"].astype(int).to_numpy()
target_names = ["No Depression", "Depression"]

# Standardization is critical for distance-based algorithms and PCA/t-SNE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original feature space dimensions: {X_scaled.shape[1]}")

## 1. Baseline Model (All 30 Features)
First, let's establish a baseline by training a k-NN classifier on the full 30-dimensional dataset.

In [ ]:
# Using k=7 as determined optimal in previous experiments
knn_baseline = KNeighborsClassifier(n_neighbors=7)
knn_baseline.fit(X_train, y_train)

y_pred_baseline = knn_baseline.predict(X_test)
acc_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline k-NN Accuracy (30 features): {acc_baseline * 100:.2f}%")

## 2. PCA: Principal Component Analysis (Reduced to 2D)
PCA projects the data onto a 2D plane that preserves the maximum amount of variance. It is fast and deterministic.

In [ ]:
# Fit PCA on training data and transform both train and test sets
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Train k-NN on 2D PCA data
knn_pca = KNeighborsClassifier(n_neighbors=7)
knn_pca.fit(X_train_pca, y_train)
y_pred_pca = knn_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"PCA (2D) k-NN Accuracy: {acc_pca * 100:.2f}%")
print(f"Explained Variance by 2 components: {sum(pca.explained_variance_ratio_) * 100:.2f}%")

# Visualization
plt.figure(figsize=(8, 6))
colors = ['#FF4136', '#0074D9'] # Red for Malignant, Blue for Benign
for color, i, target_name in zip(colors, [0, 1], target_names):
    plt.scatter(X_train_pca[y_train == i, 0], X_train_pca[y_train == i, 1], 
                color=color, alpha=0.7, lw=2, label=target_name)
plt.title('PCA of Breast Cancer Dataset (2D)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(loc='best', shadow=False, scatterpoints=1)
plt.grid(alpha=0.3)
plt.show()

## 3. t-SNE: t-Distributed Stochastic Neighbor Embedding (Reduced to 2D)
t-SNE is a non-linear technique primarily used for visualization. It maps high-dimensional distances to lower-dimensional probabilities, maintaining local cluster structures.

*Note: t-SNE does not have a standard `.transform()` method for new data. For machine learning pipelines, we fit t-SNE on the entire dataset solely to visualize and demonstrate accuracy on the transformed clusters. In practice, PCA is preferred for live inference pipelines.*

In [ ]:
# Apply t-SNE to the entire scaled dataset
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

# Split the transformed t-SNE data using the same random state to match targets
X_train_tsne, X_test_tsne, _, _ = train_test_split(X_tsne, y, test_size=0.2, random_state=42, stratify=y)

# Train k-NN on 2D t-SNE data
knn_tsne = KNeighborsClassifier(n_neighbors=7)
knn_tsne.fit(X_train_tsne, y_train)
y_pred_tsne = knn_tsne.predict(X_test_tsne)
acc_tsne = accuracy_score(y_test, y_pred_tsne)

print(f"t-SNE (2D) k-NN Accuracy: {acc_tsne * 100:.2f}%")

# Visualization
plt.figure(figsize=(8, 6))
for color, i, target_name in zip(colors, [0, 1], target_names):
    plt.scatter(X_tsne[y == i, 0], X_tsne[y == i, 1], 
                color=color, alpha=0.7, lw=2, label=target_name)
plt.title('t-SNE of Breast Cancer Dataset (2D)')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(loc='best', shadow=False, scatterpoints=1)
plt.grid(alpha=0.3)
plt.show()

## 4. Comparing Accuracy Before and After Feature Reduction
Reducing from 30 dimensions to 2 dimensions inevitably loses some information, which often causes a drop in accuracy. Let's compare how much predictive power was retained.

In [ ]:
methods = ['Original (30D)', 'PCA (2D)', 't-SNE (2D)']
accuracies = [acc_baseline * 100, acc_pca * 100, acc_tsne * 100]

plt.figure(figsize=(8, 5))
bars = plt.bar(methods, accuracies, color=['#2ECC40', '#FF851B', '#B10DC9'], alpha=0.8)

# Add accuracy labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.ylim(80, 105) # Zoom in to see the differences clearly
plt.ylabel('Test Accuracy (%)')
plt.title('k-NN Accuracy: High Dimensional vs. Reduced Dimensionality')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 2. Fit PCA and Calculate Explained Variance
We fit PCA without specifying `n_components`. This forces PCA to calculate the principal components for the entire feature space (all 30 dimensions). We then extract the `explained_variance_ratio_` to see how much information each component holds.

In [ ]:
# Fit PCA on all components
pca = PCA()
pca.fit(X_scaled)

# Get the variance explained by each individual component
individual_variance = pca.explained_variance_ratio_

# Calculate the cumulative variance
cumulative_variance = np.cumsum(individual_variance)

## 3. Scree Plot and Cumulative Variance Plot
Let's visualize the variance. 
* The **bar chart (Scree Plot)** shows the variance captured by each individual component. You'll notice the first few capture the most, and it quickly drops off (forming an "elbow").
* The **step plot (Cumulative Variance)** shows the total variance captured as we add components together. We often look for the point where this line crosses a threshold (like 90% or 95%).

In [ ]:
plt.figure(figsize=(10, 6))

# Bar chart for individual variance
plt.bar(range(1, len(individual_variance) + 1), individual_variance, alpha=0.6, align='center', 
        label='Individual Explained Variance', color='skyblue')

# Step plot for cumulative variance
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid', 
         label='Cumulative Explained Variance', color='red')

# Add threshold lines for 90% and 95% variance
plt.axhline(y=0.90, color='g', linestyle='--', label='90% Information Threshold')
plt.axhline(y=0.95, color='orange', linestyle='--', label='95% Information Threshold')

plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Components')
plt.title('PCA Explained Variance (Scree Plot)')
plt.xticks(np.arange(1, 31, step=2))
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Automating Component Selection
Instead of reading the graph visually, we can write code to programmatically find the exact number of components required to hit our desired information threshold.

In [ ]:
threshold_90 = 0.90
threshold_95 = 0.95

# Find the index where cumulative variance first exceeds the threshold (add 1 because index is 0-based)
components_90 = np.argmax(cumulative_variance >= threshold_90) + 1
components_95 = np.argmax(cumulative_variance >= threshold_95) + 1

print(f"Original number of features: {X_scaled.shape[1]}")
print(f"Number of components to retain {threshold_90*100}% of the variance: {components_90}")
print(f"Number of components to retain {threshold_95*100}% of the variance: {components_95}")

# Scikit-Learn shortcut: You can pass a float to PCA directly!
pca_shortcut = PCA(n_components=0.95)
X_pca_shortcut = pca_shortcut.fit_transform(X_scaled)
print(f"\nUsing PCA(n_components=0.95) shortcut, PCA automatically chose {pca_shortcut.n_components_} components.")

In [ ]:
# Fit PCA on training data and transform both train and test sets
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Train k-NN on 2D PCA data
knn_pca = KNeighborsClassifier(n_neighbors=7)
knn_pca.fit(X_train_pca, y_train)
y_pred_pca = knn_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

print(f"PCA (10D) k-NN Accuracy: {acc_pca * 100:.2f}%")
print(f"Explained Variance by 10 components: {sum(pca.explained_variance_ratio_) * 100:.2f}%")
